# COM299 Applied Data Science  
## Initial Inspection of the STATS19 Casualty Dataset

This notebook begins the inspection of the 2025 STATS19 casualty dataset for the SB4 Insurance Analytics Team brief.

The main purpose of this notebook is to understand the structure of the casualty data, identify the outcome variable, and begin deciding the most appropriate unit of analysis for the project.

### Load the casualty dataset

The first step is to load the raw casualty CSV into pandas.

At this stage, the raw data is only being read and inspected. No cleaning or transformation is being applied.

In [1]:
import pandas as pd

casualty_path = "../data/raw/dft-road-casualty-statistics-casualty-2025.csv"

casualties = pd.read_csv(casualty_path)

### Check the dataset size

Before analysing the data, it is useful to confirm how many rows and columns are present.

This gives a basic overview of the dataset and helps establish the scale of the casualty records being analysed.

In [2]:
casualties.shape

(127883, 24)

**Result:** The casualty dataset contains 127,883 rows and 24 columns.

This means there are 127,883 casualty records available for 2025, with 24 variables describing each record.

### Inspect the available columns

The next step is to review the column names in the casualty dataset.

This helps identify which variables describe the casualty, which fields link to collision and vehicle records, and which variable represents casualty severity.

In [3]:
casualties.columns.tolist()

['collision_index',
 'collision_year',
 'collision_ref_no',
 'vehicle_reference',
 'casualty_reference',
 'casualty_class',
 'sex_of_casualty',
 'age_of_casualty',
 'age_band_of_casualty',
 'casualty_severity',
 'pedestrian_location',
 'pedestrian_movement',
 'car_passenger',
 'bus_or_coach_passenger',
 'pedestrian_road_maintenance_worker',
 'casualty_type',
 'casualty_imd_decile',
 'lsoa_of_casualty',
 'enhanced_casualty_severity',
 'casualty_injury_based',
 'casualty_adjusted_severity_serious',
 'casualty_adjusted_severity_slight',
 'casualty_distance_banding',
 'escooter_flag']

**Result:** The casualty dataset contains fields that identify the collision, the vehicle involved, the individual casualty, and the casualty severity.

The most important fields for the current project are:

- `collision_index` / `collision_ref_no` – identify the collision
- `vehicle_reference` – links the casualty to a vehicle
- `casualty_reference` – identifies the casualty within the collision
- `casualty_severity` – the main outcome variable for SB4

These fields are important because they will help determine the correct unit of analysis and how the casualty data should later be linked to the vehicle and collision datasets.

### Check casualty-level uniqueness

To decide whether casualty-level analysis is appropriate, it is important to check whether each casualty is uniquely identified within a collision.

The combination of `collision_index` and `casualty_reference` should identify one casualty record. If no duplicates are found, this supports the idea that each row represents one unique casualty.

In [4]:
casualties.duplicated(
    subset=["collision_index", "casualty_reference"]
).sum()

np.int64(0)

**Result:** No duplicate combinations of `collision_index` and `casualty_reference` were found.

This supports the idea that each row in the casualty dataset represents one unique casualty within a collision. This is important because casualty severity is recorded at this level, making casualty-level analysis a strong candidate for the project.

### Inspect casualty class

The `casualty_class` field describes the role of the casualty in the collision.

Inspecting this variable will help clarify how casualty records relate to vehicles and whether every casualty should be treated in the same way when linking to vehicle-level factors.

In [5]:
casualties["casualty_class"].value_counts(dropna=False)

casualty_class
1    83976
2    23929
3    19978
Name: count, dtype: int64

**Result:** The `casualty_class` variable contains three coded categories.

- Code `1`: 83,976 records
- Code `2`: 23,929 records
- Code `3`: 19,978 records

The codes should not be interpreted from memory or assumption. Their meanings need to be confirmed using the official STATS19 data guide before they are used in the analysis.

**Interpretation:** According to the official STATS19/STATS20 definitions:

- Code `1` = Driver / rider
- Code `2` = Vehicle / pillion passenger
- Code `3` = Pedestrian

The largest group in the 2025 casualty dataset is drivers/riders, followed by vehicle/pillion passengers and pedestrians.

This distinction is important for the SB4 analysis because vehicle-level factors may link naturally to drivers and passengers, while pedestrian casualties need more careful consideration when vehicle information is joined.

### Check vehicle reference availability

Because many of the SB4 factors are recorded at vehicle level, it is important to check whether every casualty record has a `vehicle_reference`.

If some casualty records do not have a vehicle reference, this will affect how vehicle-level factors can be joined to the casualty data.

In [6]:
casualties["vehicle_reference"].isna().sum()

np.int64(0)

**Result:** No missing values were found in `vehicle_reference`.

This means every casualty record has a vehicle reference recorded in the casualty dataset.

For SB4, this is encouraging because it suggests casualty records can be linked to vehicle-level information without losing records simply because the reference field is missing.

### Compare vehicle references by casualty class

Although every casualty has a `vehicle_reference`, the meaning of that reference may differ depending on whether the casualty is a driver/rider, passenger, or pedestrian.

Checking the distribution of `vehicle_reference` by `casualty_class` will help us understand how vehicle linkage works across the three casualty groups.

In [7]:
casualties.groupby("casualty_class")["vehicle_reference"].nunique()

casualty_class
1    12
2    10
3     8
Name: vehicle_reference, dtype: int64

**Result:** The number of distinct `vehicle_reference` values is small within each casualty class.

However, `vehicle_reference` is not a globally unique vehicle identifier. The reference is used within an individual collision, so the same values can appear repeatedly across different collisions.

Therefore, `vehicle_reference` should not be interpreted on its own. Vehicle records will need to be linked using a combination of the collision identifier and `vehicle_reference`.

### Inspect casualty severity

The stakeholder question focuses on factors associated with casualty severity.

Before analysing any relationships, the distribution of `casualty_severity` should be inspected to understand how many records fall into each severity category.

In [8]:
casualties["casualty_severity"].value_counts(dropna=False)

casualty_severity
3    98587
2    27758
1     1538
Name: count, dtype: int64

**Result:** The `casualty_severity` variable contains three coded severity levels:

- Code `1` = Fatal: 1,538 casualties
- Code `2` = Serious: 27,758 casualties
- Code `3` = Slight: 98,587 casualties

The majority of casualties are recorded as slight, with serious casualties forming a smaller group and fatal casualties being comparatively rare.

This is important for SB4 because `casualty_severity` is the main outcome variable. The uneven distribution between the three severity categories will need to be considered later when selecting appropriate statistical methods.

### Check missing casualty severity values

Because casualty severity is the main outcome variable for the project, each casualty record needs a valid severity value.

Missing severity values could affect which records are available for later statistical analysis.

In [9]:
casualties["casualty_severity"].isna().sum()

np.int64(0)

**Result:** No missing values were found in `casualty_severity`.

This means all 127,883 casualty records have a recorded severity value, so no records would be lost later simply because the outcome variable is missing.**Result:** No missing values were found in `casualty_severity`.

This means all 127,883 casualty records have a recorded severity value, so no records would be lost later simply because the outcome variable is missing.

### Check data types for key casualty fields

Before moving further, the data types of the main identifier and outcome fields should be checked.

This helps confirm whether the variables are stored in a suitable form for later joins, validation, and analysis.

In [10]:
casualties[
    [
        "collision_index",
        "collision_ref_no",
        "vehicle_reference",
        "casualty_reference",
        "casualty_class",
        "casualty_severity"
    ]
].dtypes

collision_index         str
collision_ref_no        str
vehicle_reference     int64
casualty_reference    int64
casualty_class        int64
casualty_severity     int64
dtype: object

**Result:** The key identifier and severity fields are stored using appropriate data types.

- `collision_index` and `collision_ref_no` are stored as strings, which is appropriate because identifiers should not be treated as numeric measurements.
- `vehicle_reference`, `casualty_reference`, `casualty_class`, and `casualty_severity` are stored as integers.

No immediate data-type changes are required for these fields at this stage.

### Inspect sample casualty records

A small sample of the key fields is reviewed to check that collision, vehicle, casualty, class, and severity values appear in a sensible structure.

This is a visual sanity check before moving on to deeper validation.

In [11]:
casualties[
    [
        "collision_index",
        "collision_ref_no",
        "vehicle_reference",
        "casualty_reference",
        "casualty_class",
        "casualty_severity"
    ]
].head()

,collision_index,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,casualty_severity
0,2025010552583,010552583,1,1,3,3
1,2025010554623,010554623,1,1,3,3
2,2025010556322,010556322,1,1,3,2
3,2025010558573,010558573,1,1,3,2
4,2025010560329,010560329,1,1,3,3


**Result:** The sample records show that each casualty row contains a collision identifier, vehicle reference, casualty reference, casualty class, and casualty severity.

The identifiers and coded values appear to be structured consistently across the sample. This supports continuing with casualty-level inspection before validating the relationships with the vehicle and collision datasets separately.

### Check for multiple casualties within a collision

Although each casualty record is unique, a single collision may contain more than one casualty.

Checking this is important because it confirms that collision-level and casualty-level records are not the same thing. This helps justify why casualty severity should be analysed at casualty level rather than assuming one row represents one collision.

In [12]:
casualties["collision_index"].value_counts().head(10)

collision_index
2025052501204    142
2025041681268     32
2025520501079     25
2025351616661     22
2025440281666     20
2025340NN2152     20
2025161545000     18
2025430230402     18
2025061619139     17
2025991611220     15
Name: count, dtype: int64

### Count unique collisions

The casualty dataset contains multiple casualty records for some collisions.

Counting the number of unique collision identifiers helps distinguish the number of casualty records from the number of collisions represented in the dataset.

In [13]:
casualties["collision_index"].nunique()

101525

**Result:** The casualty dataset contains 127,883 casualty records across 101,525 unique collisions.

This confirms that some collisions involve more than one casualty. The number of casualty records is therefore larger than the number of collisions represented in the dataset.

This provides further evidence that casualty-level analysis is appropriate for SB4 because the outcome variable, `casualty_severity`, is recorded for each individual casualty rather than once per collision.

### Check for multiple casualties linked to the same vehicle

A single vehicle involved in a collision may be associated with more than one casualty.

Checking this helps confirm the relationship between casualty records and vehicle records before any joins are attempted later.

In [14]:
casualties.groupby(
    ["collision_index", "vehicle_reference"]
).size().sort_values(ascending=False).head(10)

collision_index  vehicle_reference
2025052501204    1                    142
2025041681268    1                     32
2025520501079    1                     25
2025440281666    1                     20
2025161545000    1                     18
2025430230402    1                     18
2025340NN2152    2                     17
2025061619139    1                     17
2025010595487    1                     13
2025991611220    2                     13
dtype: int64

**Result:** Some collision and vehicle combinations are linked to multiple casualty records.

For example, one `collision_index` and `vehicle_reference` combination is associated with 142 casualties.

This shows that the relationship between vehicles and casualties is not one-to-one. A single vehicle record may be linked to multiple casualty records.

For the later table joins, this suggests the casualty-to-vehicle relationship may be many-to-one when using `collision_index` together with `vehicle_reference`. This will need to be validated against the vehicle dataset before the tables are merged.

### Check missing values across the casualty dataset

Missing data can affect which variables are suitable for later analysis.

A first-pass missing-value check is used to identify which casualty fields contain blank values before any cleaning decisions are made.

In [15]:
casualties.isna().sum()

collision_index                       0
collision_year                        0
collision_ref_no                      0
vehicle_reference                     0
casualty_reference                    0
casualty_class                        0
sex_of_casualty                       0
age_of_casualty                       0
age_band_of_casualty                  0
casualty_severity                     0
pedestrian_location                   0
pedestrian_movement                   0
car_passenger                         0
bus_or_coach_passenger                0
pedestrian_road_maintenance_worker    0
casualty_type                         0
casualty_imd_decile                   0
lsoa_of_casualty                      0
enhanced_casualty_severity            0
casualty_injury_based                 0
casualty_adjusted_severity_serious    0
casualty_adjusted_severity_slight     0
casualty_distance_banding             0
escooter_flag                         0
dtype: int64

**Result:** No standard missing values (`NaN`) were found in any of the 24 casualty dataset columns.

This suggests the casualty dataset is structurally complete in terms of blank or null values.

However, this does not mean that every field contains a fully valid or meaningful value. STATS19 uses coded values, including possible sentinel values such as unknown, not recorded, or not applicable. These need to be checked against the official data definitions before the data can be considered clean.

### Inspect casualty age values

Although there are no standard missing values, coded datasets can use special numeric values to represent unknown or not recorded information.

The `age_of_casualty` field is inspected to identify whether any unusual or sentinel-style values are present.

In [16]:
casualties["age_of_casualty"].value_counts().sort_index().head(15)

age_of_casualty
-1     2741
 0      166
 1      203
 2      269
 3      385
 4      432
 5      515
 6      539
 7      560
 8      686
 9      700
 10     853
 11    1271
 12    1335
 13    1194
Name: count, dtype: int64

**Result:** The `age_of_casualty` field contains 2,741 records with a value of `-1`.

A negative age is not a valid real-world age, so this is likely to be a sentinel value representing something such as unknown or not recorded.

This confirms that the absence of `NaN` values does not mean the dataset has no missing or unavailable information. Sentinel values will need to be identified using the official STATS19 definitions before cleaning or analysis.

**Result:** The `age_of_casualty` field contains 2,741 records with a value of `-1`.

A value of `-1` is not a valid age and is used as a coded missing or unknown-style value rather than a genuine measurement.

This confirms that the absence of `NaN` values does not mean the dataset contains no missing information. Sentinel values must be identified and handled using the official STATS19/STATS20 definitions before analysis.

### Inspect casualty age bands

The `age_band_of_casualty` field groups casualties into coded age categories.

Inspecting this field helps identify whether unknown or sentinel-style categories are present and whether the grouped age variable may be useful later in the SB4 analysis.

In [17]:
casualties["age_band_of_casualty"].value_counts(dropna=False).sort_index()

age_band_of_casualty
-1      2741
 1      1970
 2      3338
 3      6271
 4     13686
 5     13021
 6     24510
 7     19748
 8     15517
 9     13397
 10     7222
 11     6462
Name: count, dtype: int64

**Result:** The `age_band_of_casualty` field contains the same 2,741 records coded as `-1` that appeared in `age_of_casualty`.

This shows that the unknown or unavailable age information is carried through into the grouped age-band variable as well.

The remaining values are coded categories rather than literal ages, so their meanings must be confirmed using the official STATS19 lookup before they are interpreted or used in analysis.

### Inspect casualty sex categories

The `sex_of_casualty` field is a coded categorical variable that may be useful later when exploring whether casualty severity differs across demographic groups.

Before using it, the distribution should be inspected to identify the available categories and any unknown or sentinel-style values.

In [18]:
casualties["sex_of_casualty"].value_counts(dropna=False).sort_index()

sex_of_casualty
-1     1320
 1    77637
 2    48874
 9       52
Name: count, dtype: int64

**Result:** The `sex_of_casualty` field contains four coded values:

- `-1`: 1,320 records
- `1`: 77,637 records
- `2`: 48,874 records
- `9`: 52 records

The values `-1` and `9` appear to represent special coded categories rather than ordinary sex categories.

The meanings of all four codes should be confirmed using the official STATS19/STATS20 lookup before the variable is interpreted or used in analysis.

**Interpretation:** The official STATS20 specification defines the main sex codes as:

- `1` = Male
- `2` = Female
- `3` = Unknown

The 2025 open dataset also contains values `-1` and `9`. These should not be interpreted until their exact meanings are confirmed from the corresponding open-data lookup or metadata.

This highlights the importance of checking coded values against the documentation rather than assuming that all codes follow the same convention.

### Inspect casualty type

The `casualty_type` field is a coded variable describing the type of road user involved.

Inspecting the distribution helps identify the available categories and whether any unknown or sentinel-style values are present before the field is interpreted or used in analysis.

In [19]:
casualties["casualty_type"].value_counts(dropna=False).sort_index()

casualty_type
-1         2
 0     19978
 1     16042
 2       732
 3      7320
 4      1662
 5      4882
 8      1599
 9     65921
 10      233
 11     2036
 16       65
 17       78
 18       16
 19     3931
 20      202
 21      401
 22      271
 23      481
 33     1227
 90      305
 97      370
 98      114
 99       15
Name: count, dtype: int64

**Result:** The `casualty_type` field contains many coded categories, with some values occurring very frequently and others only rarely.

The dataset also contains a small number of special-looking values such as `-1`, `97`, `98`, and `99`, which may represent unknown, other, or not recorded categories.

The field should not be interpreted until the official STATS19/STATS20 lookup is consulted, because each numeric code represents a specific road-user type.

### Compare casualty severity by casualty class

Casualty severity is the main outcome variable for SB4.

Comparing the raw counts of severity across casualty classes provides an initial view of how fatal, serious, and slight outcomes are distributed between drivers/riders, passengers, and pedestrians.

At this stage, this is descriptive only and does not imply causation.

In [20]:
pd.crosstab(
    casualties["casualty_class"],
    casualties["casualty_severity"]
)

casualty_severity,1,2,3
casualty_class,,,
1,960,17780,65236
2,214,4147,19568
3,364,5831,13783


**Result:** Casualty severity varies across the three casualty classes in the raw counts.

- Drivers/riders: 960 fatal, 17,780 serious, 65,236 slight
- Vehicle/pillion passengers: 214 fatal, 4,147 serious, 19,568 slight
- Pedestrians: 364 fatal, 5,831 serious, 13,783 slight

Drivers/riders have the largest number of casualties in every severity category, but they also form the largest casualty class overall.

For this reason, the raw counts alone should not be used to conclude that drivers/riders have a greater likelihood of severe injury. Later analysis should compare proportions or rates within each casualty class rather than relying only on total counts.

At this stage, the comparison is descriptive and does not establish causation.